# Efficient FunSearch - Local Demo

This notebook is aligned with the current repository implementation:
- behavioral deduplication
- diversity-guided selection
- proposal-v1 ablation registry


## 1) Environment setup (local)

In [ ]:
# Run once in a fresh env
# !pip install -e .
# !pip install -e .[dev]

In [ ]:
from src.archive import ProgramArchive
from src.integration import FunSearchAdapter, FunSearchConfig
from src.integration.ablation_configs import get_v1_ablation_configs
from src.metrics import EfficiencyTracker
from src.normalizer import ProgramNormalizer
from src.similarity import HybridSimilarityDetector

print('Imports OK')

## 2) Core module quick checks

In [ ]:
normalizer = ProgramNormalizer()
detector = HybridSimilarityDetector()

code_a = "def solve(x):\n    return x + 1"
code_b = "def solve(y):\n    return y + 1"

norm_a = normalizer.normalize(code_a)
norm_b = normalizer.normalize(code_b)
sim = detector.is_similar(norm_a, norm_b)

print('Same AST hash:', norm_a.ast_hash == norm_b.ast_hash)
print('Detector duplicate:', sim.is_duplicate)
print('Method:', sim.detection_method)

In [ ]:
archive = ProgramArchive()

p1 = "def h(x):\n    return x * 2"
p2 = "def h(y):\n    return y * 2"

n1 = normalizer.normalize(p1)
n2 = normalizer.normalize(p2)

archive.add(p1, n1, score=1.0)
print('is_duplicate(p2):', archive.is_duplicate(n2))
stats = archive.stats()
print('archive total:', stats.total_programs)
print('archive best_score:', stats.best_score)

## 3) Proposal-v1 ablation registry

In [ ]:
configs = get_v1_ablation_configs()
[c.name for c in configs]

## 4) Milestone-style smoke run (same fields as docs/milestone/reproduction.txt)

In [ ]:
import json


def eval_fn(code: str) -> float:
    return float(len(code))

initial_programs = [
    "def solve(x): return x + 1",
    "def solve(y): return y + 1",
    "def solve(x): return x + 1",
    "def solve(x): return x * 2",
]

def run(setting: str, use_dedup: bool):
    cfg = FunSearchConfig(max_generations=1, verbose=False, use_duplicate_detection=use_dedup)
    adapter = FunSearchAdapter(cfg)
    result = adapter.run(initial_programs=initial_programs, evaluation_function=eval_fn)
    return {
        "setting": setting,
        "N_total": result.total_programs_generated,
        "N_unique": result.unique_programs_evaluated,
        "sample_efficiency": result.efficiency_metrics.sample_efficiency,
        "duplicate_rate": result.efficiency_metrics.duplicate_detection_rate,
        "convergence_proxy": "1-generation smoke run",
        "final_quality_proxy": result.best_score,
    }

rows = [
    run("original", use_dedup=False),
    run("behavioral_plus_diversity", use_dedup=True),
]
print(json.dumps(rows, ensure_ascii=False, indent=2))

## 5) Optional tracker sanity check

In [ ]:
tracker = EfficiencyTracker()
for i in range(10):
    tracker.record_generation()
    if i % 3 == 0:
        tracker.record_duplicate()
        tracker.record_filtered()
    else:
        tracker.record_evaluation()

m = tracker.metrics
print('generated:', m.total_programs_generated)
print('duplicates:', m.duplicates_detected)
print('evaluated:', m.programs_evaluated)
print('sample_efficiency:', m.sample_efficiency)